# Praktikum RNN & LSTM: Prediksi Retail Sales

NAMA  : OKTOVA YOGA PRADITIA SAMOSIR<BR>
NIM   : 42324001<BR>
PRODI : D3-TEKNOLOGI INFORMASI

# 1. Persiapan Lingkungan Pengembangan

In [ ]:
!pip install -q tensorflow

cell ini memasang tensorflow yang dipakai untuk membangun model rnn dan lstm. di google colab biasanya tensorflow sudah terpasang, jadi cell ini hanya memastikan saja. opsi -q dipakai supaya output instalasinya tidak memenuhi notebook.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.dates as mdates

# TimeseriesGenerator sudah dihapus di tensorflow versi baru (keras 3),
# jadi kalau import gagal dibuat versi sederhananya sendiri
try:
    from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator
except ImportError:
    class TimeseriesGenerator(tf.keras.utils.PyDataset):
        def __init__(self, data, targets, length, batch_size=1, **kwargs):
            super().__init__(**kwargs)
            self.data = np.asarray(data, dtype='float32')
            self.targets = np.asarray(targets, dtype='float32')
            self.length = length
            self.batch_size = batch_size
            self.n = len(self.data) - length
        def __len__(self):
            return int(np.ceil(self.n / self.batch_size))
        def __getitem__(self, idx):
            rows = range(idx * self.batch_size, min((idx + 1) * self.batch_size, self.n))
            X = np.array([self.data[i:i + self.length] for i in rows])
            y = np.array([self.targets[i + self.length] for i in rows])
            return X, y

cell ini memanggil library yang akan dipakai:
- numpy dan pandas untuk mengolah data angka dan tabel.
- matplotlib dan seaborn untuk membuat grafik.
- tensorflow keras untuk membuat model simplernn dan lstm.
- minmaxscaler untuk mengubah skala data ke rentang 0 sampai 1.
- timeseriesgenerator untuk membuat pasangan input dan target dari data berurutan.
- earlystopping untuk menghentikan training kalau val_loss tidak membaik.

karena di tensorflow versi terbaru timeseriesgenerator sudah tidak ada, saya tambahkan try except. kalau import gagal, dibuat class pengganti yang cara kerjanya sama.

# 2. Unduh Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()

cell ini menampilkan tombol upload untuk mengunggah file RSCCASN.csv ke colab. dataset ini berisi data penjualan ritel bulanan dari januari 1992 sampai oktober 2019.

# 3. Pengolahan Data (Data Processing)

In [ ]:
# Membaca dataset
dataset_path = 'RSCCASN.csv'
sales_data = pd.read_csv(dataset_path, parse_dates=True, index_col='DATE')
sales_data.columns = ['Sales']

sales_data.head()

pd.read_csv() membaca file csv dengan kolom DATE dijadikan index bertipe tanggal. nama kolom RSCCASN diganti menjadi Sales supaya lebih mudah dibaca. sales_data.head() menampilkan lima baris pertama untuk mengecek isi datanya.

In [ ]:
sales_data.describe()

describe() menampilkan ringkasan statistik kolom sales, yaitu jumlah data, rata-rata, simpangan baku, nilai minimum, kuartil, dan nilai maksimum. dari sini bisa dilihat ada 334 data bulanan dan rentang nilai penjualannya cukup jauh antara bulan terendah dan tertinggi.

In [ ]:
# Plot data penjualan
sales_data.plot(figsize=(12, 8))
plt.show()

cell ini menampilkan grafik penjualan dari tahun ke tahun. terlihat penjualan punya tren naik dalam jangka panjang dan pola musiman yang berulang, dengan puncak hampir selalu di bulan desember. sekitar tahun 2008 sampai 2009 juga ada penurunan karena krisis ekonomi. pola seperti ini cocok dipelajari oleh rnn dan lstm karena keduanya bisa mengingat data dari langkah waktu sebelumnya.

In [ ]:
# Train-test split
test_size = 18
test_index = len(sales_data) - test_size
train = sales_data.iloc[:test_index]
test = sales_data.iloc[test_index:].copy()

print('data train :', len(train))
print('data test  :', len(test))

data dibagi menjadi data train dan test. karena ini data time series, data tidak diacak. 18 bulan terakhir dijadikan data test dan sisanya data train, jadi hasilnya 316 data train dan 18 data test. .copy() dipakai supaya nanti tidak muncul warning saat kolom prediksi ditambahkan ke test.

# 4. Normalisasi Data

In [ ]:
# Inisialisasi MinMaxScaler
scaler = MinMaxScaler()
scaler.fit(train)

# Transformasi data train dan test
scaled_train = scaler.transform(train)
scaled_test = scaler.transform(test)

minmaxscaler mengubah data ke rentang 0 sampai 1 supaya training model lebih stabil. scaler.fit(train) menghitung nilai minimum dan maksimum hanya dari data train, supaya informasi dari data test tidak ikut bocor ke model. setelah itu scaler.transform dipakai untuk mengubah data train dan test.

# 5. Membuat Time Series Generator

In [ ]:
# Membuat generator data time series
length = 12 # Panjang input (12 bulan)
n_features = 1
generator = TimeseriesGenerator(scaled_train, scaled_train, length=length, batch_size=1)

X, y = generator[0]
print('jumlah sampel :', len(generator))
print('input X :', X.flatten())
print('target y:', y)

timeseriesgenerator membuat pasangan input dan target dari data train. length=12 berarti model melihat 12 bulan sebelumnya untuk memprediksi 1 bulan berikutnya. 12 dipilih karena satu pola musiman penjualan terjadi dalam 12 bulan. n_features=1 karena hanya ada satu kolom, yaitu sales.

output menunjukkan contoh sampel pertama: X berisi 12 nilai penjualan pertama yang sudah dinormalisasi, dan y adalah nilai bulan ke-13 yang harus ditebak model.

# 6. Membangun Model RNN

In [ ]:
# Model SimpleRNN
model_rnn = Sequential()
model_rnn.add(SimpleRNN(100, activation='relu', input_shape=(length, n_features)))
model_rnn.add(Dense(1))
model_rnn.compile(optimizer='adam', loss='mse')
model_rnn.summary()

cell ini membangun model rnn dengan sequential:
- simplernn(100) berisi 100 unit. layer ini memproses data satu per satu sesuai urutan waktu dan menyimpan informasi langkah sebelumnya di hidden state.
- dense(1) adalah output layer yang menghasilkan satu nilai prediksi penjualan.
- model dicompile dengan optimizer adam dan loss mse.

pada summary, simplernn punya 10.200 parameter, yaitu bobot dari input ke hidden state (U), dari hidden state ke hidden state berikutnya (W), dan bias. dense punya 101 parameter (V dan bias).

In [ ]:
# Callback early stopping untuk RNN
early_stop_rnn = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Validation generator
validation_generator = TimeseriesGenerator(scaled_test, scaled_test, length=length, batch_size=1)

# Melatih model RNN
history_rnn = model_rnn.fit(generator, epochs=20, validation_data=validation_generator, callbacks=[early_stop_rnn])

model rnn dilatih maksimal 20 epoch. validation_generator dibuat dari data test untuk menghitung val_loss setiap epoch. earlystopping dengan patience=3 akan menghentikan training kalau val_loss tidak turun selama 3 epoch berturut-turut, lalu restore_best_weights=True mengembalikan bobot terbaik. output tiap epoch menampilkan loss dan val_loss, dan nilai yang makin kecil berarti prediksi makin dekat dengan data asli.

# 7. Membangun Model LSTM

In [ ]:
# Model LSTM dengan penambahan Dropout
model_lstm = Sequential()
model_lstm.add(LSTM(100, activation='relu', input_shape=(length, n_features)))
model_lstm.add(Dropout(0.2)) # Penambahan Dropout untuk mengurangi overfitting
model_lstm.add(Dense(1))
model_lstm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='mse')
model_lstm.summary()

cell ini membangun model lstm:
- lstm(100) berisi 100 unit. lstm punya cell state sebagai memori jangka panjang dan tiga gate, yaitu forget gate, input gate, dan output gate, untuk mengatur informasi mana yang dibuang, disimpan, dan dikeluarkan.
- dropout(0.2) mematikan 20% neuron secara acak saat training untuk mengurangi overfitting.
- dense(1) menghasilkan satu nilai prediksi.
- optimizer adam dengan learning rate 0.001 dan loss mse.

parameter lstm sebanyak 40.800, sekitar 4 kali lipat dari simplernn, karena lstm punya 4 set bobot (3 gate dan kandidat cell state). karena itu training lstm biasanya lebih lama.

In [ ]:
# Callback early stopping untuk LSTM
early_stop_lstm = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# Melatih model LSTM
history_lstm = model_lstm.fit(generator, epochs=20, validation_data=validation_generator, callbacks=[early_stop_lstm])

model lstm dilatih maksimal 20 epoch dengan data validasi yang sama. patience diset 5, lebih longgar dari rnn, karena lstm biasanya butuh waktu lebih lama untuk turun loss nya.

# 8. Evaluasi Model

In [ ]:
# Fungsi untuk plot training dan validation loss
def plot_loss(history_rnn, history_lstm):
    # Plot the training and validation loss
    losses_rnn = pd.DataFrame(history_rnn.history)
    losses_lstm = pd.DataFrame(history_lstm.history)

    plt.figure(figsize=(10, 6))
    plt.plot(losses_rnn['loss'], label='RNN Loss', linewidth=2)
    plt.plot(losses_rnn['val_loss'], label='RNN Val Loss', linewidth=2)
    plt.plot(losses_lstm['loss'], label='LSTM Loss', linewidth=2)
    plt.plot(losses_lstm['val_loss'], label='LSTM Val Loss', linewidth=2)
    plt.title('Training and Validation Loss', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

plot_loss(history_rnn, history_lstm)

fungsi plot_loss menyimpan riwayat loss dan val_loss dari history ke dataframe, lalu menampilkan kurva rnn dan lstm dalam satu grafik. kalau loss dan val_loss sama-sama turun, berarti model belajar dengan baik. kalau loss terus turun tapi val_loss naik, itu tanda overfitting.

In [ ]:
def plot_loss_rnn(history_rnn):
    # Plot the training and validation loss
    losses_rnn = pd.DataFrame(history_rnn.history)

    plt.figure(figsize=(10, 6))
    plt.plot(losses_rnn['loss'], label='RNN Loss', linewidth=2)
    plt.plot(losses_rnn['val_loss'], label='RNN Val Loss', linewidth=2)
    plt.title('Training and Validation Loss', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

def plot_loss_lstm(history_lstm):
    # Plot the training and validation loss
    losses_lstm = pd.DataFrame(history_lstm.history)

    plt.figure(figsize=(10, 6))
    plt.plot(losses_lstm['loss'], label='LSTM Loss', linewidth=2)
    plt.plot(losses_lstm['val_loss'], label='LSTM Val Loss', linewidth=2)
    plt.title('Training and Validation Loss', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

plot_loss_rnn(history_rnn)
plot_loss_lstm(history_lstm)

cell ini sama seperti sebelumnya, tapi grafik rnn dan lstm dipisah supaya kurva masing-masing model lebih jelas terlihat.

# 9. Prediksi Model dan Plot Prediksi

In [ ]:
# Membuat prediksi untuk data test dengan RNN dan LSTM
def make_predictions(model, scaled_data, length, n_features):
    predictions = []
    first_eval_batch = scaled_data[-length:]
    current_batch = first_eval_batch.reshape((1, length, n_features))

    for i in range(len(test)):
        current_pred = model.predict(current_batch, verbose=0)[0]
        predictions.append(current_pred)
        current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

    return scaler.inverse_transform(predictions)

rnn_predictions = make_predictions(model_rnn, scaled_train, length, n_features)
lstm_predictions = make_predictions(model_lstm, scaled_train, length, n_features)

# Menambahkan prediksi ke dataframe test
test['RNN_Predictions'] = rnn_predictions
test['LSTM_Predictions'] = lstm_predictions
test

fungsi make_predictions memprediksi data test secara bertahap:
1. ambil 12 data terakhir dari data train sebagai input awal.
2. model memprediksi 1 bulan berikutnya.
3. nilai paling lama dibuang dan hasil prediksi dimasukkan ke akhir input.
4. diulang sampai 18 bulan sesuai jumlah data test.

hasil prediksi dikembalikan ke skala aslinya dengan scaler.inverse_transform, lalu ditambahkan sebagai kolom RNN_Predictions dan LSTM_Predictions pada tabel test. verbose=0 dipakai supaya progress predict tidak muncul berulang-ulang.

In [ ]:
# Menghitung error prediksi
from sklearn.metrics import mean_absolute_error, mean_squared_error

for nama in ['RNN_Predictions', 'LSTM_Predictions']:
    mae = mean_absolute_error(test['Sales'], test[nama])
    rmse = np.sqrt(mean_squared_error(test['Sales'], test[nama]))
    print(f'{nama} -> MAE: {mae:.2f} | RMSE: {rmse:.2f}')

cell ini saya tambahkan untuk membandingkan kedua model dengan angka. mae adalah rata-rata selisih absolut prediksi dengan data asli, dan rmse adalah akar dari rata-rata kuadrat selisihnya. semakin kecil nilainya berarti prediksi model semakin mendekati penjualan sebenarnya.

In [ ]:
# Fungsi untuk plot prediksi RNN dan LSTM bersama data actual
def plot_predictions(test, sales_data, rnn_predictions, lstm_predictions):
    # Plot RNN and LSTM predictions along with actual sales
    plt.figure(figsize=(14, 8))
    plt.plot(test.index, test['Sales'], label='Actual Sales', color='blue', linewidth=2)
    plt.plot(test.index, test['RNN_Predictions'], label='RNN Predictions', color='orange', linestyle='--', linewidth=2)
    plt.plot(test.index, test['LSTM_Predictions'], label='LSTM Predictions', color='green', linestyle='--', linewidth=2)

    # Formatting the x-axis to show dates nicely
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.gcf().autofmt_xdate() # Rotate the x labels for better readability

    plt.title('Actual vs RNN and LSTM Predictions', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Sales', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.xlim([test.index[0], test.index[-1]])
    plt.show()

plot_predictions(test, sales_data, rnn_predictions, lstm_predictions)

cell ini menampilkan grafik perbandingan penjualan asli (biru) dengan prediksi rnn (oranye putus-putus) dan prediksi lstm (hijau putus-putus) selama periode test. sumbu x diformat per 3 bulan supaya tanggalnya mudah dibaca. semakin dekat garis prediksi dengan garis biru, semakin bagus modelnya.

# 10. Prediksi Jangka Panjang

In [ ]:
# Seluruh data dinormalisasi supaya forecast mulai dari data terakhir (okt 2019)
scaled_full = scaler.transform(sales_data[['Sales']])

# Membuat prediksi ke depan dengan model
def forecast_future(model, scaled_data, length, n_features, periods):
    forecast = []
    first_eval_batch = scaled_data[-length:]
    current_batch = first_eval_batch.reshape((1, length, n_features))

    for i in range(periods):
        current_pred = model.predict(current_batch, verbose=0)[0]
        forecast.append(current_pred)
        current_batch = np.append(current_batch[:, 1:, :], [[current_pred]], axis=1)

    return scaler.inverse_transform(forecast)

# Prediksi untuk 12 periode ke depan
forecast_rnn = forecast_future(model_rnn, scaled_full, length, n_features, periods=12)
forecast_lstm = forecast_future(model_lstm, scaled_full, length, n_features, periods=12)

# Membuat dataframe untuk forecast
forecast_index = pd.date_range(start='2019-11-01', periods=12, freq='MS')
forecast_rnn_data = pd.DataFrame(data=forecast_rnn, index=forecast_index, columns=['RNN_Forecast'])
forecast_lstm_data = pd.DataFrame(data=forecast_lstm, index=forecast_index, columns=['LSTM_Forecast'])

pd.concat([forecast_rnn_data, forecast_lstm_data], axis=1)

fungsi forecast_future cara kerjanya sama dengan make_predictions, tapi dipakai untuk memprediksi 12 bulan ke depan, yaitu november 2019 sampai oktober 2020. hasilnya disimpan ke dataframe forecast_rnn_data dan forecast_lstm_data.

di modul, input awalnya memakai scaled_train, jadi forecast sebenarnya mulai dari akhir data train (april 2018). di sini saya ganti dengan scaled_full (seluruh data yang sudah dinormalisasi) supaya forecast benar-benar melanjutkan dari data terakhir, yaitu oktober 2019.

In [ ]:
# Fungsi untuk plot forecast ke depan
def plot_forecast(sales_data, forecast_rnn_data, forecast_lstm_data):
    # Mendapatkan nilai dan index terakhir dari sales_data
    last_actual_value = sales_data['Sales'].iloc[-1]
    last_actual_date = sales_data.index[-1]

    # Menyusun prediksi agar dimulai tepat setelah data actual
    forecast_rnn_data.index = pd.date_range(start=last_actual_date + pd.DateOffset(months=1), periods=len(forecast_rnn_data), freq='MS')
    forecast_lstm_data.index = pd.date_range(start=last_actual_date + pd.DateOffset(months=1), periods=len(forecast_lstm_data), freq='MS')

    # Menambahkan nilai terakhir data actual sebagai titik awal prediksi
    forecast_rnn_data = pd.concat([pd.DataFrame({'RNN_Forecast': [last_actual_value]}, index=[last_actual_date]), forecast_rnn_data])
    forecast_lstm_data = pd.concat([pd.DataFrame({'LSTM_Forecast': [last_actual_value]}, index=[last_actual_date]), forecast_lstm_data])

    # Plot actual sales dan prediksi forecast
    plt.figure(figsize=(14, 8))

    # Plot actual sales
    plt.plot(sales_data.index, sales_data['Sales'], label='Actual Sales', color='blue', linewidth=2)

    # Plot RNN forecast
    plt.plot(forecast_rnn_data.index, forecast_rnn_data['RNN_Forecast'],
             label='RNN Forecast', color='orange', linestyle='--', linewidth=2)

    # Plot LSTM forecast
    plt.plot(forecast_lstm_data.index, forecast_lstm_data['LSTM_Forecast'],
             label='LSTM Forecast', color='green', linestyle='--', linewidth=2)

    # Set the limits for x-axis to control the date range
    plt.xlim(pd.to_datetime('2018-01-01'), pd.to_datetime('2020-12-01'))

    # Formatting the x-axis to show dates nicely
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    plt.gca().xaxis.set_major_locator(mdates.MonthLocator(interval=3)) # Show every 3 months
    plt.gcf().autofmt_xdate()

    plt.title('Sales Forecast using RNN and LSTM', fontsize=16)
    plt.xlabel('Date', fontsize=12)
    plt.ylabel('Sales', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True)
    plt.show()

# Contoh pemanggilan fungsi setelah model dijalankan
plot_forecast(sales_data, forecast_rnn_data, forecast_lstm_data)

fungsi plot_forecast menampilkan data penjualan asli bersama hasil forecast rnn dan lstm untuk 12 bulan ke depan. index forecast disesuaikan supaya mulai tepat satu bulan setelah data terakhir, dan nilai asli terakhir ditambahkan di awal supaya garis forecast nyambung dengan garis data asli. grafik dibatasi dari januari 2018 sampai desember 2020 supaya bagian forecast terlihat jelas.

karena prediksinya dilakukan bertahap (hasil prediksi dipakai lagi sebagai input), error nya bisa makin besar semakin jauh bulan yang diprediksi. jadi hasil forecast lebih tepat dibaca sebagai gambaran pola penjualan ke depan.